In [1]:
import pandas as pd
from helpers import get_factor, get_price
pd.options.mode.chained_assignment = None

In [2]:
CDF = pd.read_csv("../production-v2/CDF.csv")
raw_sse = pd.read_csv("../basic/inspire_prtr_mapper.csv")
see = raw_sse.rename(columns={"InspireID_Betrieb": "plantid"})
seem = see[['plantid', 'sseid']]
#bpm['plantid'] = bpm['plantid'].apply(lambda x: str(x).replace('/', '_'))
seem['plantid'] = seem['plantid'].apply(lambda x: str(x).replace('/', '_'))

In [3]:
#CDF.sort_values(by=["produced_at", "variable"])

In [4]:
smard = pd.read_csv("Gro_handelspreise_202301010000_202401010000_Viertelstunde.csv", sep=";", na_values="-", decimal=",", thousands=".")
smardlog = smard[["Datum von", "Deutschland/Luxemburg [€/MWh] Originalauflösungen"]]
smardlog.rename(columns={"Datum von": "timestamp", "Deutschland/Luxemburg [€/MWh] Originalauflösungen": "price"}, inplace=True)
smardlog["timestamp"] = pd.to_datetime(smardlog["timestamp"], format="mixed")

In [5]:
co2s = pd.read_csv("../pollution/pollutants.csv")
nat_mp = pd.read_csv("nat_mapper_2025.csv")
plantlist = pd.read_csv("../basic/plants_2.csv")
nat_mp.fillna(0, inplace=True)

In [6]:
CDF2 = CDF.loc[CDF.produced_at > "2022-12-31 23:50"].loc[CDF.produced_at < "2024-01-01 00:00"]

In [7]:
CDF3 = CDF2.loc[~(CDF.variable == "Unnamed: 3")]

In [8]:
#CDF3 = CDF2.dropna()

In [9]:
len(CDF2) - len(CDF3)

35036

In [10]:
len(CDF2.groupby('produced_at').sum())

35036

In [11]:
dataset = CDF3.merge(seem, left_on="variable", right_on="sseid")

In [12]:
magic = dataset.groupby(["produced_at", "plantid"]).sum()

In [13]:
#magic

In [14]:
magic2 = magic[["value"]]

In [15]:
#magic2.sort_values(["produced_at", "value"])

In [16]:
production = magic2.reset_index()
production["produced_at"] = pd.to_datetime(production["produced_at"], format="mixed")

In [17]:
merged = production.merge(smardlog, left_on="produced_at", right_on="timestamp")
merged["revenue"] = merged["value"] * merged["price"]

In [18]:
#merged

In [19]:
#production.sort_values(by=["plantid", "produced_at"])

In [20]:
#merged1a

In [21]:

#merged_tmp = merged.drop(columns=["timestamp"])
#merged1a = merged_tmp.set_index("produced_at", drop=True)


In [22]:
#merged1a = merged_tmp.set_index(['plantid']).sort_values(['plantid', 'produced_at'])

In [23]:
#merged2 = merged1a.resample("1h", on="produced_at").agg({'value':'sum', 'price':'sum', 'revenue': 'sum' })

In [24]:
#merged1a

In [25]:
#tmp1 = merged2.copy()
tmp1 = merged[["plantid", "value", "price", "revenue"]].groupby("plantid").sum()

In [26]:
tmp1.reset_index(inplace=True)

In [27]:
revenue = tmp1[["plantid", "revenue"]]

In [28]:
tmp1

,plantid,value,price,revenue
0,06-02-B10117A007,707521.0,833736.96,6.933753e+07
1,BB23020490,1163264.0,3334947.84,1.106459e+08
2,BB45025564,11579284.0,3334947.84,1.180307e+09
3,BB45025611,8443056.0,833736.96,8.770335e+08
4,BE166928,1269720.0,3334947.84,1.269227e+08
5,BE169709,347644.0,833736.96,3.511908e+07
6,BE172654,1365724.0,3334947.84,1.427438e+08
7,BE172656,1135173.0,833736.96,1.200585e+08
8,BWpf-450-1020129-00000000,306092.0,3334947.84,4.390331e+07
9,BWpf-450-1195689-00000000,186372.0,3334947.84,1.825098e+07


In [29]:
#dataset.sort_values(["plantid", "produced_at"])

In [30]:
plantlist2 = plantlist[["plantid", "energysource"]]
plantlist3 = plantlist2.merge(nat_mp, on="plantid")

In [31]:
tmp0 = pd.merge(revenue, plantlist3, on="plantid")
tmp0["factor"] = tmp0["energysource"].apply(get_factor)
tmp0["fuel_price"] = tmp0["energysource"].apply(get_price)

In [32]:
#prod2 = prod.loc[prod.year == 2023].loc[prod.yearpower > 1000000]
co2s2 = co2s.loc[co2s.year == 2023].loc[co2s.pollutant == "CO2"]

In [33]:
co2s2.drop_duplicates(subset=["year", "plantid", "pollutant"], inplace=True)

In [34]:
co2s2

,year,plantid,pollutant,releases_to,amount,potency,unit_2,amount_2,pollutant2
52,2023,BB16018798,CO2,Air,2.850000e+08,9,Mio. t,0.285,CO2 [Mio. t]
84,2023,BB23020389,CO2,Air,4.340000e+08,9,Mio. t,0.434,CO2 [Mio. t]
163,2023,BB23020490,CO2,Air,3.015000e+09,9,Mio. t,3.015,CO2 [Mio. t]
369,2023,BB23022811,CO2,Air,1.500000e+08,9,Mio. t,0.150,CO2 [Mio. t]
411,2023,BB45025564,CO2,Air,1.413400e+10,9,Mio. t,14.134,CO2 [Mio. t]
...,...,...,...,...,...,...,...,...,...
15336,2023,ST18046,CO2,Air,2.210000e+08,9,Mio. t,0.221,CO2 [Mio. t]
15459,2023,TH30013152,CO2,Air,2.620000e+08,9,Mio. t,0.262,CO2 [Mio. t]
15490,2023,TH62013494,CO2,Air,1.290000e+08,9,Mio. t,0.129,CO2 [Mio. t]
15566,2023,TH72012874,CO2,Air,1.770000e+08,9,Mio. t,0.177,CO2 [Mio. t]


In [35]:
co2s3 = co2s2[["plantid", "amount_2"]]

In [36]:
tmp1 = pd.merge(tmp0, co2s3, on="plantid")

In [37]:
tmp2 = pd.merge(tmp1, production, on="plantid")

In [38]:
tmp2 = tmp1

In [39]:
coal_cost_per_t = 103.5# or 120
co2_cost = 70
#electricity_price = 78.50

In [40]:
tmp2["co2_cost"] = (tmp2["amount_2"] * 10**6 - tmp2["free_co2s"]) * co2_cost / 10**6
tmp2["coal_cost"] = (tmp2["amount_2"] * 10**6 * 1/tmp2["factor"] * tmp2["fuel_price"]) / 10**6

In [41]:
tmp2["profit"] = (tmp2["revenue"] / 10**6) - (tmp2["co2_cost"] + tmp2["coal_cost"])

In [42]:
#tmp2.sort_values(by="free_co2s", ascending=False)

In [43]:
profit = tmp2[["plantid", "plantname", "revenue", "profit"]]
profit["revenue"] = profit["revenue"].apply(lambda x: x / 10**6)

In [44]:
#profit.to_csv("profit.csv", index=False)

In [46]:
profit.sort_values("profit", ascending=True)

,plantid,plantname,revenue,profit
42,RP5000671,GuD Mitte DT10,218.220516,-265.692818
0,BB23020490,1MKA,110.645937,-198.719281
12,BWpf-450-2948214-00000000,GKM Block 6,247.873328,-139.366872
35,NW500-0342658,Scholven 1 DT,118.681204,-106.696332
11,BWpf-450-2797933-00000000,Rheinhafen- Dampfkraftwerk RDK 4S DT,265.822193,-99.739748
14,BYS00041,SWM HKW Nord 2 T20,113.533971,-88.499362
20,MV30000226,Kraftwerk Rostock Block A,135.462265,-61.858110
9,BWpf-450-1741292-00000000,Heizkraftwerk Altbach/Deizisau HKW 1,56.354601,-46.208461
3,BE166928,HKW Reuter West Dampfturbine D,126.922714,-46.158883
22,NI01241117210,DT30 GuD Süd Wolfsburg,118.842134,-45.172941
